# Week 2 - ML2 - Exercise 2: Simple RAG From PDFs

You are building a tournament prep assistant for a fictional esports coaching staff.
The assistant must answer strategy questions from internal team reports and clearly signal uncertainty when evidence is weak.

In this exercise, RAG is new. We build it step by step with minimal abstractions.

Dependencies used in this notebook include:
- `faiss-cpu`
- `openai`
- `langchain` (for `RecursiveCharacterTextSplitter`)
- `PyPDF2`, `python-dotenv`

You will:
1. Load PDFs
2. Chunk text
3. Retrieve relevant chunks
4. Generate grounded answers
5. Test failure cases and improve one step

Edit only cells marked `TODO`.


## Learning Goals

- Understand core RAG components
- Use a standard out-of-the-box text splitter
- Build retrieval with embeddings + FAISS (`faiss-cpu`)
- Compare before/after one improvement


In [2]:
import os
from collections import Counter
from pathlib import Path

import faiss
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()


True

In [3]:
DATA_DIR = Path("data")
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

if not PDF_FILES:
    raise FileNotFoundError("No PDFs found in week2/data. Add PDF files first.")

PDF_FILES

[WindowsPath('data/esports_overview_01_phantom_signal_valorant.pdf'),
 WindowsPath('data/esports_overview_02_iron_tide_cs2.pdf'),
 WindowsPath('data/esports_overview_03_neon_assembly_valorant.pdf'),
 WindowsPath('data/esports_overview_04_vector_praetorians_cs2.pdf'),
 WindowsPath('data/esports_overview_05_midnight_orbit_valorant.pdf')]

## Step 1 - Read PDFs

In [5]:
def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages).strip()


documents = []
for path in PDF_FILES:
    text = read_pdf_text(path)
    documents.append(
        {
            "source": path.name,
            "text": text,
            "char_count": len(text),
        }
    )

documents[:2]


incorrect startxref pointer(1)
incorrect startxref pointer(1)
incorrect startxref pointer(1)
incorrect startxref pointer(1)
incorrect startxref pointer(1)


[{'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'text': 'Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling\nThis overview describes a fictional esports team created for teaching purposes.\nPhantom Signal started the split looking like a team assembled from highlight clips rather than\nshared habits. Their early matches were loud and frantic, with heroic moments that disguised\nweak structure. Round reviews showed the same pattern every series: good opening reads, poor\nmid-round adaptation, and visible emotional spikes after lost advantage rounds. The coaches did\nnot panic about tactics first. They treated communication discipline as the first lever, arguing\nthat no playbook survives if players stop hearing each other when pressure rises.\nThe in-game leader, Riven, had inherited an identity that was too scripted for the roster she\nactually had. In previous teams she won with set executions and tightly sequenced utility, but\nthis 

## Step 2 - Chunk Text (`TODO 1`)

In [6]:
# TODO 1
# 1) Set CHUNK_SIZE and CHUNK_OVERLAP

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunk_records = []
for doc in documents:
    chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunk_records.append(
            {
                "source": doc["source"],
                "chunk_id": f"{doc['source']}_chunk_{i}",
                "chunk_text": chunk,
            }
        )

chunk_records[:10]


[{'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_0',
  'chunk_text': 'Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling\nThis overview describes a fictional esports team created for teaching purposes.\nPhantom Signal started the split looking like a team assembled from highlight clips rather than\nshared habits. Their early matches were loud and frantic, with heroic moments that disguised\nweak structure. Round reviews showed the same pattern every series: good opening reads, poor\nmid-round adaptation, and visible emotional spikes after lost advantage rounds. The coaches did\nnot panic about tactics first. They treated communication discipline as the first lever, arguing'},
 {'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_1',
  'chunk_text': 'not panic about tactics first. They treated c

In [8]:
chunk_count_by_source = Counter(item["source"] for item in chunk_records)

print("Number of documents:", len(documents))
print("Number of chunks:", len(chunk_records))
print("\nChunks per source:")
for source, count in chunk_count_by_source.items():
    print(f"- {source}: {count}")


Number of documents: 5
Number of chunks: 41

Chunks per source:
- esports_overview_01_phantom_signal_valorant.pdf: 8
- esports_overview_02_iron_tide_cs2.pdf: 9
- esports_overview_03_neon_assembly_valorant.pdf: 8
- esports_overview_04_vector_praetorians_cs2.pdf: 8
- esports_overview_05_midnight_orbit_valorant.pdf: 8


## Step 3 - Build FAISS Retriever


### 3.1 Embed all chunks once

We first create embeddings for all chunks and store them in memory.
This pattern is model-agnostic: later you can swap embedding models without changing the overall flow.


In [9]:
embedding_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
EMBEDDING_MODEL = "text-embedding-3-small"


def embed_texts(texts, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = embedding_client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch,
        )
        vectors.extend(item.embedding for item in response.data)
    return np.array(vectors, dtype="float32")


chunk_texts = [item["chunk_text"] for item in chunk_records]
chunk_embeddings = embed_texts(chunk_texts)
faiss.normalize_L2(chunk_embeddings)
print("Chunk embeddings shape:", chunk_embeddings.shape)


Chunk embeddings shape: (41, 1536)


### 3.2 Build the vector index

FAISS stores vectors and lets us search nearest neighbors efficiently.
Here we use inner product on normalized vectors, which behaves like cosine similarity.


In [10]:
faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)
print("Vectors stored in FAISS:", faiss_index.ntotal)


Vectors stored in FAISS: 41


### 3.3 Create a retrieval function

At query time, we embed the query with the same model, search the index, and map hits back to chunk metadata.
This same function design works for other vector stores too.


In [11]:
def retrieve_chunks(query: str, top_k: int = 3):
    query_response = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )
    query_vector = np.array([query_response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vector)

    k = min(top_k, len(chunk_texts))
    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = chunk_records[int(idx)]
        results.append(
            {
                "source": item["source"],
                "chunk_id": item["chunk_id"],
                "similarity": float(score),
                "chunk_text": item["chunk_text"],
            }
        )

    return results


In [12]:
# TODO 2
# 1) Set TOP_K
# 2) Test retrieval with two meaningful questions
# 3) Add a short comment: Are top chunks relevant?
# Note: For cosine/IP similarity with FAISS, higher is better.

TOP_K = 3

q1 = "How did Phantom Signal improve after early losses?"
q2 = "What role did communication protocols play in Neon Assembly's turnaround?"

for query in [q1, q2]:
    print("=" * 80)
    print("Query:", query)
    results = retrieve_chunks(query, top_k=TOP_K)
    for r in results:
        print(f"- {r['chunk_id']} | similarity={r['similarity']:.4f}")
        print(f"  {r['chunk_text'][:180]}...")

# Comment:
# ...


Query: How did Phantom Signal improve after early losses?
- esports_overview_01_phantom_signal_valorant.pdf_chunk_6 | similarity=0.6250
  decision mattered more than the last headline.
By playoffs, Phantom Signal looked fundamentally different without appearing revolutionary. They
did not suddenly gain impossible mec...
- esports_overview_01_phantom_signal_valorant.pdf_chunk_0 | similarity=0.6158
  Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling
This overview describes a fictional esports team created for teaching purposes.
Phantom Signal ...
- esports_overview_01_phantom_signal_valorant.pdf_chunk_7 | similarity=0.6114
  Their strategic outlook now rests on three principles. First, tactical depth grows only after
communication reliability is stable. Second, role comfort and team trust are co-depend...
Query: What role did communication protocols play in Neon Assembly's turnaround?
- esports_overview_03_neon_assembly_valorant.pdf_chunk_0 | similarity

## Step 4 - Prompt (`TODO 3`)

In [13]:
# TODO 3
# Write a prompt that:
# - uses only provided context
# - is clear and short
# - says: "I don't know based on the provided context." if context is insufficient

PROMPT_TEMPLATE = """
You are the tournament prep assistant for a fictional esports coaching staff.
Use ONLY the provided context from internal team reports.
Focus on strategic points (communication, leadership, adaptation, preparation routines, and risk areas).
Keep the answer concise and practical for coaches.
If the context is insufficient, say exactly: "I don't know based on the provided context."

Context:
{context}

Question:
{question}
"""

## Step 5 - Retrieval + Generation

In [14]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL_NAME = "gpt-5.4-nano"


def generate_answer(question: str, top_k: int = 3):
    retrieved = retrieve_chunks(question, top_k=top_k)
    context = "\n\n".join(item["chunk_text"] for item in retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )
    answer = response.output_text
    return answer, retrieved


In [15]:
test_question = "What strategic changes helped Iron Tide recover from bad starts?"
answer, used_chunks = generate_answer(test_question, top_k=TOP_K)

print("Question:", test_question)
print("\nAnswer:\n", answer)
print("\nRetrieved Chunks:")
for item in used_chunks:
    print(f"- {item['chunk_id']} | similarity={item['similarity']:.4f}")


Question: What strategic changes helped Iron Tide recover from bad starts?

Answer:
 Iron Tide recovered from bad starts mainly through **composure + more information-based communication**, and **disciplined mid-round decision-making**:

- **Communication stayed functional** and mid-round calls reflected **available information** instead of **emotional urgency**, preventing spirals after poor openings.
- They adopted a clearer **strategic profile**: **flexible defaults**, **disciplined transitions**, and **only selectively going sharp with set pieces** once opponent reads were validated.
- Their practice/anti-strat work shifted to **behavioral, scenario-based goals** (e.g., deny default comfort early, force utility imbalances before first executes, punish predictable saves), so players could meet the plan even when scorelines were noisy.
- After a specific collapse from **map point** due to **rushed mid-round calls**, review directly targeted the recovery problem: players were trying t

## Step 6 - Failure Cases (`TODO 4`)

Test:
1. Vague query (for example: `Tell me more.`)
2. Out-of-scope query

In [16]:
# TODO 4
vague_query = "Tell me more."
out_of_scope_query = "How can I optimize my home Wi-Fi router placement?"

for q in [vague_query, out_of_scope_query]:
    ans, ret = generate_answer(q, top_k=TOP_K)
    print("=" * 80)
    print("Question:", q)
    print("Answer:", ans)
    print("Top retrieved chunk IDs:", [item["chunk_id"] for item in ret])

# Short note (3-5 lines):
# What failed and why?


Question: Tell me more.
Answer: I don't know based on the provided context.
Top retrieved chunk IDs: ['esports_overview_01_phantom_signal_valorant.pdf_chunk_0', 'esports_overview_05_midnight_orbit_valorant.pdf_chunk_7', 'esports_overview_04_vector_praetorians_cs2.pdf_chunk_5']
Question: How can I optimize my home Wi-Fi router placement?
Answer: I don't know based on the provided context.
Top retrieved chunk IDs: ['esports_overview_04_vector_praetorians_cs2.pdf_chunk_1', 'esports_overview_02_iron_tide_cs2.pdf_chunk_5', 'esports_overview_05_midnight_orbit_valorant.pdf_chunk_1']


## Step 7 - One Improvement (`TODO 5`)

Choose one:
- Option A: query rewrite for vague follow-up queries
- Option B: prompt refinement for stricter groundedness

In [17]:
# TODO 5 - Option A starter

def rewrite_query_if_vague(query: str, last_user_query: str):
    vague_patterns = ["tell me more", "more", "what about that", "and this"]
    q = query.lower().strip()
    if q in vague_patterns:
        return f"Provide more details about: {last_user_query}"
    return query


def generate_answer_with_rewrite(question: str, last_user_query: str, top_k: int = 3):
    rewritten = rewrite_query_if_vague(question, last_user_query)
    answer, retrieved = generate_answer(rewritten, top_k=top_k)
    return rewritten, answer, retrieved

In [19]:
# TODO 5 - Evaluate before vs after

test_cases = [
    {
        "question": "Tell me more",
        "last_user_query": "How did Midnight Orbit stabilize communication after coaching changes?",
    },
    {
        "question": "And this",
        "last_user_query": "Why did Vector Praetorians simplify their strategy book?",
    },
]

records = []
for case in test_cases:
    q = case["question"]
    last_q = case["last_user_query"]

    base_answer, _ = generate_answer(q, top_k=TOP_K)
    rewritten_q, improved_answer, _ = generate_answer_with_rewrite(q, last_user_query=last_q, top_k=TOP_K)

    records.append(
        {
            "original_query": q,
            "rewritten_query": rewritten_q,
            "baseline_answer_preview": base_answer[:180],
            "improved_answer_preview": improved_answer[:180],
        }
    )

for row in records:
    print("=" * 80)
    print("Original query:", row["original_query"])
    print("Rewritten query:", row["rewritten_query"])
    print("Baseline preview:", row["baseline_answer_preview"])
    print("Improved preview:", row["improved_answer_preview"])

# Conclusion (2-4 lines):
# Did your improvement help? Why?


Original query: Tell me more
Rewritten query: Provide more details about: How did Midnight Orbit stabilize communication after coaching changes?
Baseline preview: I don't know based on the provided context.
Improved preview: I don't know based on the provided context.
Original query: And this
Rewritten query: Provide more details about: Why did Vector Praetorians simplify their strategy book?
Baseline preview: I don't know based on the provided context.
Improved preview: Vector Praetorians simplified their strategy book because their coaches found a recurring, decision-level failure pattern:

- **They treated prep as content volume, not decision qu


## Submission Checklist

- [ ] Chunking choice documented
- [ ] Retrieval tested with at least 2 meaningful queries
- [ ] Grounded prompt implemented
- [ ] 2 failure cases analyzed
- [ ] 1 improvement implemented and compared
- [ ] Final reflection completed

## Final Reflection (max 8 lines)

1. What worked best in your pipeline?
2. What was the biggest failure mode?
3. Which design choice mattered most?
4. One responsible-use limitation of this setup.